In [14]:
# Load env variables
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
from anthropic import Anthropic

client=Anthropic()
# model="claude-opus-5"
model = "claude-haiku-4-5-20251001"

In [16]:
def add_user_message(messages, text):
    user_message={"role":"user", "content":text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message={"role":"assistant", "content":text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=None, stop_sequences=[]):
    params={
        "model":model,
        "max_tokens":1000,
        "messages":messages
    }

    #We only add system and temperature as parameters if they are defined
    if system is not None:
        params["system"]=system

    if temperature is not None:
            params["temperature"]=temperature

    if stop_sequences:
                params["stop_sequences"]=stop_sequences

    message=client.messages.create(
        **params
    )
    for block in message.content:
        if block.type=='text':
            return block.text
        

In [18]:
messages=[]

add_user_message(messages,"Generate a very short EventBridge rule as a JSON")
add_assistant_message(messages, "```json")

answer=chat(messages, stop_sequences=["```"])



In [19]:

answer

'\n{\n  "Name": "MyRule",\n  "EventBusName": "default",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["running"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",\n      "Id": "1"\n    }\n  ]\n}\n'

In [21]:
import json

clean_text=json.loads(answer.strip())

In [22]:
clean_text

{'Name': 'MyRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}